# Flickr API 

## What are API's ?

![API](https://media.geeksforgeeks.org/wp-content/uploads/20230216170349/What-is-an-API.png)

An API (Application Programming Interface) is a set of rules that allows different software programs to talk to each other.

You can think of it like a menu in a restaurant: the menu lists the dishes you can order, along with a description. When you tell the waiter what you want, the kitchen (the system behind the API) prepares it and sends it back to you.

In the same way, an API defines what requests you can make and what responses you’ll get—for example, asking a weather API for today’s temperature or a Flickr API for recent photos.

APIs are used everywhere: they let your app get data from a website, send messages, control devices, or connect services together.

## What is Flickr?

![Flickr](https://www.flickrhelp.com/hc/article_attachments/4419907628308/unnamed__1_.png)  

Flickr is an online platform for sharing, storing, and discovering photos and videos. It was one of the first major social networks built around photography.

Users can:

- Upload their own images and organize them into albums.
- Add tags, titles, and descriptions to make photos searchable.
- Explore images from other users around the world.
- Comment, like, and join photography groups or communities.

Flickr is especially popular among photographers, artists, and researchers because it provides high-quality image hosting and rich metadata (like geolocation, upload time, and camera info).

A typical Flickr post can be viewed here: [link](https://www.flickr.com/photos/92959567@N00/227080829).  

It also offers a simple public API, which allows developers to access that photo data programmatically — for example, to download geotagged images, analyze tags, or visualize where and what people photograph.

The objective of this exercise is to create a table containing images from a specific area along with their associated information.  

To achieve this, we will use the "Python Flickr API" library to handle authentication. Once authenticated, you can utilize the methods detailed in the API documentation [here](https://www.flickr.com/services/api/).  

The API methods fall under different authentication levels, such as:  
- *Requires 'write' permission for authentication.*  
- *Requires 'read' permission for authentication.*  
- *Does not require authentication.*  

Focus only on the methods that do not require authentication and disregard those needing "write" or "read" permissions.



## Geo Search
### Preparation & First Searches
First, we pip install the library with `pip install flickrapi`.

Import the passwords: 

(Dropbox oddities: Rarely, a cloud-synced placeholder can stall I/O. Ensure passwords.py is fully local (right-click in Dropbox → Make available offline).))

In [5]:
import sys
sys.path.append(r'C:\Users\VincentNowak\Dropbox\Codes\passwords')

import passwords as pw

api_key = pw.flickr_key
api_secret = pw.flickr_secret

#print (api_key)
#print (api_secret)

Import the libraries we need. The code connects to the flickr api. The connection will be stored in an object called `flickr`.

In [ ]:
import flickrapi
import pandas as pd
import folium


#connect to flickr
flickr = flickrapi.FlickrAPI(api_key, api_secret, format='parsed-json')


### Define search location and display map

In [9]:
# Define the location (latitude and longitude) and search parameters
centre_latitude = 51.51235 
centre_longitude = -0.11720 

# Define the bounding box. Use Open Street Map to get the coordinates
lat_north = 51.51491
lat_south = 51.50893
long_west = -0.12167
long_east = -0.11171

# Define the number of photos to retrieve per page
per_page = 2

# Create a map centered on the location
map_london = folium.Map(
    location=[centre_latitude, centre_longitude],
    tiles="Cartodb dark_matter",
    zoom_start=16,
    control_scale=True,
    zoom_control=False,
    dragging=False,
    scrollWheelZoom=False
)

# Add a circle marker to the map
folium.CircleMarker(
    location=[centre_latitude, centre_longitude],
    radius=2,
    color="cornflowerblue",
    stroke=False,
    fill=True,
    fill_opacity=0.6,
    opacity=1,
    # popup="{} pixels".format(radius),    
).add_to(map_london) 

# Add a rectangle to the map
folium.Rectangle(
    bounds=[[lat_north , long_west], [lat_south, long_east]],
    fill=True,
    fill_opacity=0.1,
    weight=1,
    color="cornflowerblue",  
).add_to(map_london)

# you can also save the map as html
# map_london.save("map_london.html")

map_london

The main command will be `flickr.photos.search` , please read the [documentation](https://www.flickr.com/services/api/flickr.photos.search.html) well. We will search the flickr photos that are inside the rectangle. 

In [ ]:
# Define the bounding box coordinates (latitude and longitude)
# Format: bbox = "min_longitude,min_latitude,max_longitude,max_latitude"
bbox = str(long_west)+','+str(lat_south)+','+str(long_east)+','+str(lat_north)

# Search for photos in the bounding box
photos = flickr.photos.search(bbox=bbox, 
                              per_page=per_page, 
                              page=1,
                              has_geo=1, 
                              extras='geo,description,tags,views,media,url_n,date_taken,owner_name')
# Print the results
print(photos)


{'photos': {'page': 1, 'pages': 34110, 'perpage': 2, 'total': 68220, 'photo': [{'id': '54860692981', 'owner': '99923531@N00', 'secret': 'fd28beac9c', 'server': '65535', 'farm': 66, 'title': 'Keeley Street at Wild Street', 'ispublic': 1, 'isfriend': 0, 'isfamily': 0, 'description': {'_content': 'Work underway October 2025'}, 'datetaken': '2025-10-17 12:24:46', 'datetakengranularity': 0, 'datetakenunknown': '0', 'ownername': 'Camden Cyclists', 'views': '12', 'tags': '', 'latitude': '51.514691', 'longitude': '-0.120117', 'accuracy': '16', 'context': 0, 'place_id': '', 'woeid': '2646781', 'geo_is_public': 1, 'geo_is_contact': 0, 'geo_is_friend': 0, 'geo_is_family': 0, 'media': 'photo', 'media_status': 'ready', 'url_n': 'https://live.staticflickr.com/65535/54860692981_fd28beac9c_n.jpg', 'height_n': 240, 'width_n': 320}, {'id': '54860945433', 'owner': '99923531@N00', 'secret': 'f2809e62d8', 'server': '65535', 'farm': 66, 'title': 'Keeley Street at Wild Street', 'ispublic': 1, 'isfriend': 0, 

### Parsing the results, JSON format

We have now an object called `photo` that holds the response from flickr. The respose is a text file that comes in JSON format. Similar to a csv file, it can be read by humans and machines. 

JSON is widely used and the basic structure is this:

```python
{ "name": "Zophie",
  "isCat": true,
  "miceCaught": 0,
  "napsTaken": 37.5,
  "felineIQ": null}

```
Notice the similarity to a python dict object:
```python
{ "brand": "Ford",
  "model": "Mustang",
  "year": 1964 }
```
As a reminder, a python list looks like this: 
```python
[
"Car 1", 
"Car 2", 
"Car 3"
]
```
As you might suspect, JSON Objects can be arraged in arrays: 
```python
[
{ "brand": "BMW",
  "model": "Z1",
  "year": 1995 } , 
{"brand": "Ford",
  "model": "Mustang",
  "year": 1964 }, 
{"brand": "VW",
  "model": "Polo",
  "year": 2020 }
]
```
it also goes the other way: 
```python
{ 
"brand" : ["BMW", "Ford", "VW"]
"model" : ["Z1", "Mustang", "Polo"]
"year"  : [1995, 1964, 2020]
}
```

Understanding nextled JSON data is not always straightforward. The [**pprint module**](https://docs.python.org/3/library/pprint.html) can help you to print the data in a more readable format.

In [12]:
# Print the results
import pprint

pprint.pprint(photos, compact=True)

{'photos': {'page': 1,
            'pages': 34110,
            'perpage': 2,
            'photo': [{'accuracy': '16',
                       'context': 0,
                       'datetaken': '2025-10-17 12:24:46',
                       'datetakengranularity': 0,
                       'datetakenunknown': '0',
                       'description': {'_content': 'Work underway October '
                                                   '2025'},
                       'farm': 66,
                       'geo_is_contact': 0,
                       'geo_is_family': 0,
                       'geo_is_friend': 0,
                       'geo_is_public': 1,
                       'height_n': 240,
                       'id': '54860692981',
                       'isfamily': 0,
                       'isfriend': 0,
                       'ispublic': 1,
                       'latitude': '51.514691',
                       'longitude': '-0.120117',
                       'media': 'photo',
        

The items in the tree can get accessed through "chaining". 

In [ ]:
# pprint.pprint(photos["photos"], compact=True)
# pprint.pprint(photos["photos"]["page"], compact=True)
# pprint.pprint(photos["photos"]["photo"], compact=True)
# pprint.pprint(photos["photos"]["photo"][0], compact=True)
# pprint.pprint(photos["photos"]["photo"][0]["id"], compact=True)


1
'54860692981'


### Getting all pages (Pagination)
Currently, we face two issues:  
- The response displays the results per page, in our case we can increase to a maximum of 250 results per page, but we still have more than one page.   
- The response data needs to be transformed into a table.  

The suggested solution involves the following steps:  

1. Determine the total number of pages.  
2. Retrieve each page sequentially.  
3. For every page retrieved, append the relevant content to a new table.  

### Determine the number of pages

Determine the number of pages, extact the results of the first entry.

In [21]:
photos = flickr.photos.search(bbox=bbox, 
                              per_page=per_page, 
                              page=1,
                              has_geo=1, 
                              extras='geo,description,tags,views,media,url_o,url_s,date_taken,owner_name')

total_pages = photos['photos']['pages']
total_photos = photos['photos']['total']

print(f"Total photos: {total_photos}")
print(f"Total pages: {total_pages}")
print("First entry:")
print("ID: " + photos['photos']['photo'][0]["id"])
print("Title: " + photos['photos']['photo'][0]["title"])
print("Owner: " + photos['photos']['photo'][0]["owner"])
print("Secret: " + photos['photos']['photo'][0]["secret"])
print("Server: " + photos['photos']['photo'][0]["server"])


Total photos: 76846
Total pages: 38423
First entry:
ID: 54862579584
Title: london-england_54297126517_o
Owner: 200455905@N08
Secret: 7ac03de6c5
Server: 65535


### Extract Page Information

Now, the code below is a function that takes a page, extracts the information and returns a dataframe:

In [ ]:
def get_page(bbox, page, per_page):
    response = flickr.photos.search(
        bbox=bbox,
        per_page=per_page,
        page=page,
        has_geo=1,
        extras="geo,description,tags,views,media,url_s,date_taken,owner_name",
    )
    photos = response["photos"]["photo"]
    rows = []
    for photo in photos:
        new_row = {
            "id": photo["id"],
            "server": photo["server"],
            "secret": photo["secret"],
            "title": photo["title"],
            "tags": photo["tags"],
            "views": photo["views"],
            "description": photo["description"]["_content"],
            "date_taken": photo["datetaken"],
            "latitude": photo["latitude"],
            "longitude": photo["longitude"],
            "url_s": photo["url_s"],
            "owner": photo["owner"],
            "owner_name": photo["ownername"],
            "media": photo["media"],
        }
        rows.append(new_row)
    df = pd.DataFrame(
        rows,
        columns=[
            "id",
            "server",
            "secret",
            "title",
            "tags",
            "views",
            "description",
            "date_taken",
            "latitude",
            "longitude",
            "url_s",
            "owner_name",
            "owner",
            "media",
        ],
    )
    return df


get_page(bbox, 1, per_page)

,id,server,secret,title,tags,views,description,date_taken,latitude,longitude,url_s,owner_name,owner,media
0,54289861886,65535,f3da063106,,,1,,2024-01-13 17:02:45,51.511208,-0.119756,https://live.staticflickr.com/65535/5428986188...,Ben Sutherland,60179301@N00,photo
1,54288972792,65535,97fb1e90fe,,,1,,2024-01-13 15:16:49,51.509880,-0.117000,https://live.staticflickr.com/65535/5428897279...,Ben Sutherland,60179301@N00,photo
2,54288471461,65535,eb9425fe5e,"Theatre Royal, Drury Lane",january 2025 london drury lane theatre royal,75,,2025-01-25 14:17:17,51.512836,-0.120473,https://live.staticflickr.com/65535/5428847146...,looper23,98587546@N00,photo
3,54288471376,65535,2f177f5364,"Theatre Royal, Drury Lane",january 2025 london drury lane theatre royal,76,,2025-01-25 14:17:22,51.512833,-0.120456,https://live.staticflickr.com/65535/5428847137...,looper23,98587546@N00,photo
4,54287588692,65535,8b28fddd99,"Theatre Royal, Drury Lane",january 2025 london drury lane theatre royal,73,,2025-01-25 14:17:20,51.512836,-0.120473,https://live.staticflickr.com/65535/5428758869...,looper23,98587546@N00,photo
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
245,54202471353,65535,5bf9e89d45,,,10,,2024-05-18 12:19:52,51.512408,-0.120875,https://live.staticflickr.com/65535/5420247135...,newshamyen,201772084@N07,photo
246,54200789654,65535,5ef5c5fd56,London 2024,city london night thames uk 2024 united kingdo...,5326,,2024-12-10 18:34:01,51.509149,-0.117180,https://live.staticflickr.com/65535/5420078965...,m-child,44865365@N05,photo
247,54198739687,65535,4abf2a36a5,Where To Now,london uk street candid streetphotography temp...,833,"Temple Place, City Of London",2024-11-20 12:03:25,51.511059,-0.114551,https://live.staticflickr.com/65535/5419873968...,jaykay72.,39901710@N05,photo
248,54198296937,65535,f6a543fa2c,Rotten,piano worn old wood keys rotten mobile googlep...,527,I've been a bit busy the last weeks or so and ...,2024-11-06 11:44:03,51.513675,-0.119803,https://live.staticflickr.com/65535/5419829693...,Kate H2011,67721796@N02,photo


### Go through all pages
The next step involves creating a loop that processes all the pages and consolidates the data into a single large dataframe. Since each request takes approximately 5 to 6 seconds, processing all 299 pages could take around 30 minutes. While this duration is manageable, there are potential risks during such a lengthy process.

Here are a few considerations to address these challenges:
- **Download in batches**: For instance, process 20 pages at a time. However, this may not be an ideal solution.
- **Use a smaller bounding box**: This approach is more effective.

In the example provided below, only the first 20 pages are downloaded. To process all pages, replace the relevant line with `for page in range(1, total_pages + 1):`.

In [ ]:
df = pd.DataFrame()

for page in range(1, 10):
    new_df = get_page(bbox, page, per_page)
    print(f"Getting page {page} of {total_pages}")
    df = pd.concat([df, new_df], ignore_index=True)
    print(f"Total photos so far: {len(df)}")


df

Getting page 1 of 301
Total photos so far: 250
Getting page 2 of 301
Total photos so far: 500
Getting page 3 of 301
Total photos so far: 750
Getting page 4 of 301
Total photos so far: 1000
Getting page 5 of 301
Total photos so far: 1250
Getting page 6 of 301
Total photos so far: 1500
Getting page 7 of 301
Total photos so far: 1750
Getting page 8 of 301
Total photos so far: 2000
Getting page 9 of 301
Total photos so far: 2250


,id,server,secret,title,tags,views,description,date_taken,latitude,longitude,url_s,owner_name,owner,media
0,54289861886,65535,f3da063106,,,1,,2024-01-13 17:02:45,51.511208,-0.119756,https://live.staticflickr.com/65535/5428986188...,Ben Sutherland,60179301@N00,photo
1,54288972792,65535,97fb1e90fe,,,1,,2024-01-13 15:16:49,51.509880,-0.117000,https://live.staticflickr.com/65535/5428897279...,Ben Sutherland,60179301@N00,photo
2,54288471461,65535,eb9425fe5e,"Theatre Royal, Drury Lane",january 2025 london drury lane theatre royal,75,,2025-01-25 14:17:17,51.512836,-0.120473,https://live.staticflickr.com/65535/5428847146...,looper23,98587546@N00,photo
3,54288471376,65535,2f177f5364,"Theatre Royal, Drury Lane",january 2025 london drury lane theatre royal,76,,2025-01-25 14:17:22,51.512833,-0.120456,https://live.staticflickr.com/65535/5428847137...,looper23,98587546@N00,photo
4,54287588692,65535,8b28fddd99,"Theatre Royal, Drury Lane",january 2025 london drury lane theatre royal,73,,2025-01-25 14:17:20,51.512836,-0.120473,https://live.staticflickr.com/65535/5428758869...,looper23,98587546@N00,photo
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2245,53551338017,65535,25e843ecfe,UK - London Flickr Group Photowalk 16 - Somers...,uklondonflickrgroupphotowalk16somersethousenel...,1943,"<a href=""http://www.dgphotos.co.uk"" rel=""noref...",2024-02-10 16:32:50,51.510685,-0.117470,https://live.staticflickr.com/65535/5355133801...,Darrell Godliman,79986881@N00,photo
2246,53554638852,65535,7efacfc00b,IMG_1140,,90,,2024-02-27 11:38:09,51.510494,-0.121412,https://live.staticflickr.com/65535/5355463885...,Wanderer 30,51036032@N08,photo
2247,53555774849,65535,b7d9fc8376,"Two Temple Place, London",twotempleplace london jlpearson victorian arch...,226,"The former Astor House, now known as Two Templ...",2024-01-25 19:37:41,51.511647,-0.112320,https://live.staticflickr.com/65535/5355577484...,Aidan McRae Thomson,24141292@N02,photo
2248,53555774179,65535,a59845a97d,"Glass Ceiling, Two Temple Place, London",twotempleplace london jlpearson victorian arch...,267,"The former Astor House, now known as Two Templ...",2024-01-25 20:25:33,51.511687,-0.112341,https://live.staticflickr.com/65535/5355577417...,Aidan McRae Thomson,24141292@N02,photo


In [ ]:
df[5:15]

,id,server,secret,title,tags,views,description,date_taken,latitude,longitude,url_s,owner_name,owner,media
5,54288709504,65535,2016d45a08,"Theatre Royal, Drury Lane",january 2025 london drury lane theatre royal,56,,2025-01-25 14:16:35,51.512827,-0.120445,https://live.staticflickr.com/65535/5428870950...,looper23,98587546@N00,photo
6,54286814839,65535,24060b7317,Nell of Old Drury pub,january 2025 london pub nell drury lane,100,,2025-01-25 14:09:32,51.512741,-0.120625,https://live.staticflickr.com/65535/5428681483...,looper23,98587546@N00,photo
7,54286575216,65535,7c74e7243e,Sigourney Weaver in the Tempest,january 2025 london drury lane theatre tempest...,100,,2025-01-25 13:02:21,51.512561,-0.120050,https://live.staticflickr.com/65535/5428657521...,looper23,98587546@N00,photo
8,54284962470,65535,821a2d56db,The World Turned Upside Down (2019),globe wereldbol lse londonschoolofeconomics br...,1599,Work of Art by Mark Wallinger at Saw Swee Hock...,2025-01-13 13:27:13,51.514630,-0.117568,https://live.staticflickr.com/65535/5428496247...,just.Luc,9619972@N08,photo
9,54285450403,65535,6b49896cee,UK - London - Covent Garden - Theatre Royal Dr...,uk england london coventgarden theatre theatre...,18,January 2025.\nVisit to theatre.,2025-01-23 14:08:23,51.513066,-0.120303,https://live.staticflickr.com/65535/5428545040...,JulesFoto,55578087@N08,photo
10,54285448303,65535,9d95fc25c4,UK - London - Covent Garden - Theatre Royal Dr...,uk england london coventgarden theatre theatre...,20,January 2025.\nVisit to theatre.,2025-01-23 14:08:09,51.513066,-0.120303,https://live.staticflickr.com/65535/5428544830...,JulesFoto,55578087@N08,photo
11,54281542431,65535,7a2516f577,London.,,11,,2025-01-14 18:59:12,51.511750,-0.117539,https://live.staticflickr.com/65535/5428154243...,ronindunedin,20724457@N00,photo
12,54281542416,65535,564ba2e622,London.,,13,,2025-01-14 18:52:53,51.511591,-0.117531,https://live.staticflickr.com/65535/5428154241...,ronindunedin,20724457@N00,photo
13,54281971775,65535,a9d29d4f08,London.,,11,,2025-01-14 19:04:24,51.511680,-0.117581,https://live.staticflickr.com/65535/5428197177...,ronindunedin,20724457@N00,photo
14,54281791983,65535,77ac2f5251,London.,,9,,2025-01-14 19:17:57,51.511866,-0.117645,https://live.staticflickr.com/65535/5428179198...,ronindunedin,20724457@N00,photo


and we save the dataframe....

In [ ]:
# Replace newline characters with a space or any other character
df = df.replace('\n', ' ', regex=True)

df.to_csv("london_photos2.csv", sep='\t', index=True)

### Map Display
We display the location of our points on the map: 

In [ ]:
# Create a map centered on the location
map_london = folium.Map(
    location=[centre_latitude, centre_longitude],
    tiles="Cartodb dark_matter",
    zoom_start=16,
    control_scale=True,
    zoom_control=False,
    dragging=False,
    scrollWheelZoom=False
)

# Add a circle marker to the map
folium.CircleMarker(
    location=[centre_latitude, centre_longitude],
    radius=2,
    color="cornflowerblue",
    stroke=False,
    fill=True,
    fill_opacity=0.6,
    opacity=1,
    # popup="{} pixels".format(radius),    
).add_to(map_london) 

# Add a rectangle to the map
folium.Rectangle(
    bounds=[[lat_north , long_west], [lat_south, long_east]],
    fill=True,
    fill_opacity=0.05,
    weight=.5,
    color="cornflowerblue",  
).add_to(map_london)

# Add a circle marker for each photo
latitudes = df['latitude']
longitudes = df['longitude']

for latitude, longitude in  zip(latitudes,longitudes):
  coordinate = [latitude,longitude]
  radius = 1
  folium.CircleMarker(
    location=coordinate,
    radius=radius,
    stroke=False,
    fill=True,
    fillColor="orchid", 
    fill_opacity=0.3,
    opacity=0.3,
  ).add_to(map_london)
  
map_london

**Colors**  

Ever wondered where color names like "orchid" or "cornflowerblue" come from? These are built-in Windows colors, and you can find the complete list [here](https://learn.microsoft.com/en-us/dotnet/api/system.windows.media.colors?view=windowsdesktop-9.0).

---

## Combined Script

This is the combined script: 

### Import Libaries

In [115]:
import sys

sys.path.append(r"C:\Users\vince\Dropbox\Codes\passwords")

import passwords as pw
import flickrapi
import pandas as pd
import pprint
import folium

api_key = pw.flickr_key
api_secret = pw.flickr_secret

# connect to flickr
flickr = flickrapi.FlickrAPI(api_key, api_secret, format="parsed-json")

### Set search parameters

In [126]:
# Define the location (latitude and longitude) and search parameters
centre_latitude = 51.51239
centre_longitude = 0.00496

# Define the bounding box. Use Open Street Map to get the coordinates
lat_north = 51.51550
lat_south = 51.50941
long_west = 0.00024
long_east = 0.01015

# Format: bbox = "min_longitude,min_latitude,max_longitude,max_latitude" for Flickr search
bbox = str(long_west)+','+str(lat_south)+','+str(long_east)+','+str(lat_north)

# Define the number of photos to retrieve per page
per_page = 250

# Create a map centered on the location
map_london = folium.Map(
    location=[centre_latitude, centre_longitude],
    tiles="Cartodb dark_matter",
    zoom_start=16,
    control_scale=True,
    zoom_control=False,
    dragging=False,
    scrollWheelZoom=False,
)

# Add a circle marker to the map
folium.CircleMarker(
    location=[centre_latitude, centre_longitude],
    radius=2,
    color="cornflowerblue",
    stroke=False,
    fill=True,
    fill_opacity=0.6,
    opacity=1,
    popup="{} pixels".format(radius),
).add_to(map_london)

# Add a rectangle to the map
folium.Rectangle(
    bounds=[[lat_north, long_west], [lat_south, long_east]],
    fill=True,
    fill_opacity=0.1,
    weight=1,
    color="cornflowerblue",
).add_to(map_london)

map_london

### Get amount of pages and photos

In [127]:
photos = flickr.photos.search(bbox=bbox, per_page=per_page, page=1, has_geo=1)

total_pages = photos["photos"]["pages"]
total_photos = photos["photos"]["total"]

print(f"Total photos: {total_photos}")
print(f"Total pages: {total_pages}")

Total photos: 3841
Total pages: 16


### Extract information

In [128]:
def get_page(bbox, page, per_page):
    response = flickr.photos.search(
        bbox=bbox,
        per_page=per_page,
        page=page,
        has_geo=1,
        extras="geo,description,tags,views,media,url_s,date_taken,owner_name",
    )
    photos = response["photos"]["photo"]
    rows = []
    for photo in photos:
        new_row = {
            "id": photo["id"],
            "server": photo["server"],
            "secret": photo["secret"],
            "title": photo["title"],
            "tags": photo["tags"],
            "views": photo["views"],
            "description": photo["description"]["_content"],
            "date_taken": photo["datetaken"],
            "latitude": photo["latitude"],
            "longitude": photo["longitude"],
            "url_s": photo["url_s"],
            "owner": photo["owner"],
            "owner_name": photo["ownername"],
            "media": photo["media"],
        }
        rows.append(new_row)
    df = pd.DataFrame(
        rows,
        columns=[
            "id",
            "server",
            "secret",
            "title",
            "tags",
            "views",
            "description",
            "date_taken",
            "latitude",
            "longitude",
            "url_s",
            "owner_name",
            "owner",
            "media",
        ],
    )
    return df


df = pd.DataFrame()

# Use this code to get only the first 10 pages of photos
for page in range(1, 10):
    new_df = get_page(bbox, page, per_page)
    print(f"Getting page {page} of {total_pages}")
    df = pd.concat([df, new_df], ignore_index=True)
    print(f"Total photos so far: {len(df)}")

""" 
Use this code to get all the photos in the bounding box: 
for page in range(1, total_pages + 1):
    new_df = get_page(bbox, page, per_page)
    print(f"Getting page {page} of {total_pages}")
    df = pd.concat([df, new_df], ignore_index=True)
    print(f"Total photos so far: {len(df)}") 
"""

df.to_csv("london_photos.csv", index=False)

df

Getting page 1 of 16
Total photos so far: 250
Getting page 2 of 16
Total photos so far: 500
Getting page 3 of 16
Total photos so far: 750
Getting page 4 of 16
Total photos so far: 1000
Getting page 5 of 16
Total photos so far: 1250
Getting page 6 of 16
Total photos so far: 1500
Getting page 7 of 16
Total photos so far: 1750
Getting page 8 of 16
Total photos so far: 2000
Getting page 9 of 16
Total photos so far: 2250


,id,server,secret,title,tags,views,description,date_taken,latitude,longitude,url_s,owner_name,owner,media
0,54188656132,65535,9eb07f8473,Go-Ahead London MHV6,goahead london mhv6 bu16oyo volvo b5lh mcv evo...,485,Go-Ahead London MHV6 (BU16 OYO)\nVolvo B5LH/MC...,2024-12-07 13:17:20,51.514731,0.007992,https://live.staticflickr.com/65535/5418865613...,gbenviro200,33732381@N04,photo
1,54188288630,65535,1e94c66157,EastLondon-37556-YX60DXO-CanningTown-300124,yx60dxo enviro200 firstcapital dm44169 route30...,433,East London 37556 (YX60 DXO) \n\nADL Enviro 20...,2024-01-30 00:00:00,51.514291,0.008368,https://live.staticflickr.com/65535/5418828863...,Michael Wadman,33075566@N08,photo
2,54186949737,65535,fbf67379c4,EastLondon-47992-YJ12GVR-CanningTown-300124,yj12gvr ctplus optaresolo optare solo route309...,470,East London 47992 (YJ12 GVR) \n\nFormer CT Plu...,2024-01-30 00:00:03,51.514291,0.008368,https://live.staticflickr.com/65535/5418694973...,Michael Wadman,33075566@N08,photo
3,54188115424,65535,6b76e2d37b,EastLondon-64202-LF20XKP-CanningTown-300124,lf20xkp bydd8ur route323 eastlondonbus canning...,400,East London 64202 (LF20 XKP) \n\nBYD D8UR / AD...,2024-01-30 00:00:02,51.514291,0.008368,https://live.staticflickr.com/65535/5418811542...,Michael Wadman,33075566@N08,photo
4,54186949517,65535,a17c975acf,EastLondon-47985-YJ60PFE-CanningTown-300124,yj60pfe ctplus optaresolo optare solo route309...,309,East London 47985 (YJ60 PFE) \n\nFormer CT Plu...,2024-01-30 00:00:01,51.514291,0.008368,https://live.staticflickr.com/65535/5418694951...,Michael Wadman,33075566@N08,photo
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2245,19669717713,270,d30ac971ea,"City Island, Orchard Place",roundtower towerhamlets trinitybuoywharf round...,892,Thousands of private residents will be driving...,2015-07-11 11:03:51,51.510772,0.005171,https://live.staticflickr.com/270/19669717713_...,diamond geezer,36101699310@N01,photo
2246,20261193506,459,b17fea2926,Stagecoach London 19795 on route 5 at Canning ...,bus transport publictransport stagecoach londo...,2204,This is the closest route 5 gets to Central Lo...,2015-08-03 13:57:59,51.515259,0.007756,https://live.staticflickr.com/459/20261193506_...,SW11simon,42037567@N08,photo
2247,20276112171,3788,146972c94d,City Island,roundtower towerhamlets roundtower2,1237,One day this red footbridge will be a new way ...,2015-08-02 13:49:56,51.513363,0.005578,https://live.staticflickr.com/3788/20276112171...,diamond geezer,36101699310@N01,photo
2248,19644544533,3774,b8596fbdbc,City Island,roundtower towerhamlets roundtower2,1081,"Former industrial peninsula, soon to be luxury...",2015-08-02 13:48:49,51.513363,0.005578,https://live.staticflickr.com/3774/19644544533...,diamond geezer,36101699310@N01,photo


### Map display

In [130]:
# Create a map centered on the location
map_london = folium.Map(
    location=[centre_latitude, centre_longitude],
    tiles="Cartodb dark_matter",
    zoom_start=16,
    control_scale=True,
    zoom_control=False,
    dragging=False,
    scrollWheelZoom=False
)

# Add a circle marker to the map
folium.CircleMarker(
    location=[centre_latitude, centre_longitude],
    radius=2,
    color="cornflowerblue",
    stroke=False,
    fill=True,
    fill_opacity=0.6,
    opacity=1,
    popup="{} pixels".format(radius),    
).add_to(map_london) 

# Add a rectangle to the map
folium.Rectangle(
    bounds=[[lat_north , long_west], [lat_south, long_east]],
    fill=True,
    fill_opacity=0.05,
    weight=.5,
    color="cornflowerblue",  
).add_to(map_london)

# Add a circle marker for each photo
latitudes = df['latitude']
longitudes = df['longitude']

for latitude, longitude in  zip(latitudes,longitudes):
  coordinate = [latitude,longitude]
  radius = 1
  folium.CircleMarker(
    location=coordinate,
    radius=radius,
    stroke=False,
    fill=True,
    fillColor="orchid", 
    fill_opacity=0.3,
    opacity=0.3,
  ).add_to(map_london)
  
map_london

## What's Next?  

If you examine this [Flickr Image](https://www.flickr.com/photos/33075566@N08/54187832141), you'll notice additional information associated with the image, such as "Favorites" or "Comments," which were not captured during the initial search.  
To address this, you'll need to iterate through the list and include the missing entries. Refer to the documentation to determine the appropriate API calls required to retrieve this data.

You can also convert the image's capture date into a datetime object, which helps you better analyze and understand the time aspect.